In [0]:
## Config Notebook

### Name
`Bronze_Config_ImdbAdvancedMoviesDetails.sql`

### Purpose
This notebook acts as a configuration unit for the pipeline.

It contains the SQL logic (mainly INSERT statements) required for the dataset.  
The orchestration layer loops through config notebooks and executes them dynamically.

### Why this design
- keeps logic modular  
- avoids hardcoding in orchestration  
- makes it easy to add new datasets  
- improves maintainability

In [0]:
Bronze_Config_ImdbAdvancedMoviesDetails = f"""
INSERT OVERWRITE bronze_landing.media_analytics.bronzeImdbAdvancedMoviesDetails
SELECT 
  link                                       AS link,
  writers                                    AS writers,
  directors                                  AS directors,
  stars                                      AS stars,
  budget                                     AS budget,
  opening_weekend_Gross                      AS openingWeekendGross,
  grossWorldWWide                            AS grossWorldwide,
  gross_US_Canada                            AS grossUsCanada,
  CASE
  WHEN release_date RLIKE '^[0-9]{4}$'
       THEN to_date(concat(release_date, '-01-01'))
  ELSE try_to_date(release_date, 'MMMM d, yyyy')
END                                          AS releaseDate,  -- convert only when source is mmmm d, yyyy
  countries_origin                           AS countriesOrigin,
  filming_locations                          AS filmingLocations,
  production_company                         AS productionCompany,
  awards_content                             AS awardsContent,
  genres                                     AS genres,
  Languages                                  AS languages,
  CAST(regexp_extract(_metadata.file_path, '/Data/([0-9]{4})/', 1) AS INT) AS fileYear,
  _metadata.file_path                        AS sourceFilePath,
  current_timestamp()                        AS ingestTs
FROM read_files(
  '/Volumes/bronze_landing/media_analytics/media_analytics_volume/Data/*/advanced_movies_details_*.csv',
  format => 'csv',
  header => true,
  inferSchema => false,
  multiLine => true,
  quote => '"',
  escape => '"'
)
where CAST(regexp_extract(_metadata.file_path, '/Data/([0-9]{4})/', 1) AS INT)
BETWEEN 2010 AND 2025
--LIMIT 10;
"""
dbutils.notebook.exit(Bronze_Config_ImdbAdvancedMoviesDetails)